# 💳 Credit Card Fraud Detector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/credit-card-fraud/training/notebook.ipynb)

Flags fraudulent card transactions in the [ULB credit-card dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud):
284,807 European card transactions from September 2013, of which only 492 (0.17 %) are fraud.
Twelve models are compared: plain baselines, **NearMiss** undersampling, **SMOTE** oversampling and a
**Keras MLP**. The one with the best **validation PR-AUC** is deployed, its decision threshold is tuned for
max F1 on validation, and the held-out **test** split is scored once at the end. The result is the
[model repo](https://huggingface.co/shalev396/credit-card-fraud) that the [Space](https://huggingface.co/spaces/shalev396/credit-card-fraud) serves.

This notebook is the controller: every step below calls a function in `src/` (reusable code) or
`../model/model.py` (the inference code shared with the Space and the endpoint).

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "credit-card-fraud"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
# 1d. Project code: importing `src` puts ../model on sys.path, so `import model` is model/model.py
%matplotlib inline
import time

import pandas as pd
from IPython.display import display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config
import model as M

print(utils.lib_versions("sklearn", "imblearn", "tensorflow", "keras", "numpy", "pandas"))

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` runs the same steps on a 20k-row stratified
subset with small models (0.5-2 min on CPU) and writes to `outputs/smoke/model/` instead of `../model/`.

In [ ]:
# "0" = full run (last run: 15.6 min on a shared 6-thread CPU; OVERWRITES ../model), "1" = smoke run.
# setdefault keeps a value set outside (e.g. `SMOKE_TEST=1 jupyter nbconvert --execute ...`).
os.environ.setdefault("SMOKE_TEST", "0")
PUSH_TO_HUB = False  # upload the exported model repo to the Hub at the end (section 8)

cfg = Config()
utils.setup_tf()                       # GPU memory growth when a GPU exists (float32 everywhere)
utils.set_seeds(cfg.seed, "tensorflow")
print("export to:", cfg.model_dir)
cfg

## 3. Data
`creditcard.csv` (~150 MB) is downloaded once into `data/` with kagglehub (OpenML `data_id=1597` as fallback).
`V1..V28` are PCA components (the only form the bank released), `Time` is seconds since the first
transaction, `Amount` is the value in EUR and `Class` is the label (1 = fraud). Exact duplicate rows are
dropped so that no transaction can land in two splits, then the rows are split 60/20/20 (stratified).

In [ ]:
df = data_setup.load_dataframe(cfg)                  # download -> cache -> dedupe (-> subset when smoke)
splits = data_setup.split(df, cfg)                   # stratified train / val / test
print(f"{len(df):,} unique transactions, {df[M.TARGET].sum():,} frauds ({df[M.TARGET].mean():.3%})")
display(pd.DataFrame({name: {"rows": len(part), "frauds": int(part[M.TARGET].sum())}
                      for name, part in splits.items()}).T)
df.head()

The imbalance is the whole story: predicting "legit" for every row is 99.8 % accurate and catches no fraud,
so accuracy is useless here. Models are judged by **PR-AUC** (average precision) and F1 / precision / recall.

In [ ]:
display(export.plot_eda(df))
df.groupby(M.TARGET)["Amount"].describe().round(2)

Preprocessing is `model.to_features`, the function the Space runs: `V1..V28` as-is, `Amount` and `Time`
scaled by a `RobustScaler` fit on the **train** rows only.

In [ ]:
scaler, x, y = engine.prepare_features(splits)
{name: arr.shape for name, arr in x.items()}

## 4. Load model
Eleven scikit-learn / imbalanced-learn variants (`model_builder.build_variants`; resampling sits inside an
imblearn `Pipeline`, so it only touches the rows a model is fit on) plus the Keras MLP, whose architecture
is `model.build_mlp` in `../model/model.py`.

In [ ]:
variants = model_builder.build_variants(cfg)
mlp = engine.compile_mlp(model_builder.build_mlp(cfg), cfg)
mlp.summary()
print(f"{len(variants)} scikit-learn variants + {model_builder.MLP_NAME} ({utils.count_params(mlp):,} parameters)")
pd.Series({name: model_builder.describe(name, est) for name, est in variants.items()}, name="estimator").to_frame()

## 5. Training
Every candidate is fit on the **train** split. The MLP uses balanced class weights and early stopping on a
stratified slice of the *train* split, so the validation split stays untouched for model selection.

In [ ]:
start = time.perf_counter()
fitted = engine.fit_variants(variants, x, y)
history = engine.fit_mlp(mlp, x["train"], y["train"], cfg)
fitted[model_builder.MLP_NAME] = mlp
train_time_s = time.perf_counter() - start
print(f"all {len(fitted)} candidates fit in {train_time_s:.1f} s")
display(export.plot_mlp_history(history.history))

## 6. Evaluation
For each candidate the decision threshold is set to maximise F1 on **validation**; the default model is the
one with the highest **validation PR-AUC** (`engine.select`). The test split never influences a choice: its
metrics are reported once, at each model's own tuned threshold.

In [ ]:
evals = engine.evaluate_all(fitted, x, y)
best, exported = engine.select(evals)
engine.comparison_table(evals).round(4)

In [ ]:
chosen = evals[best]
display(pd.DataFrame({"validation": chosen.val, "test": chosen.test}).round(4))
print("threshold:", round(chosen.threshold, 4), "| test confusion:",
      engine.confusion(y["test"], chosen.test_prob, chosen.threshold))
figs = export.figures(evals, best, exported, splits, history.history)
for name in ("model_comparison", "pr_curve", "threshold_tuning", "confusion_matrix"):
    display(figs[name])

## 7. Inference
Save the selected models (`export.save_models`: scaler, `model.joblib`, `model.keras`, `config.json`), reload
them through `model.load(dir, device).predict(...)` exactly like the Space, and call the Inference Endpoint
handler (`handler.EndpointHandler`) on the same data. The examples in `../space/examples` are real rows from
the held-out test split; the model ignores their `Class` column.

In [ ]:
model_dir = export.save_models(cfg, scaler=scaler, fitted=fitted, exported=exported, best=best, evals=evals)
export.check_roundtrip(model_dir, evals, splits)       # served predictions == in-memory predictions

predictor = M.load(model_dir, "cpu")
example = utils.SPACE_DIR / "examples" / "mixed_transactions.csv"
for choice in predictor.choices:
    print(choice, {k: v for k, v in predictor.predict(example, model=choice).items() if k != "rows"})
pred = predictor.predict(example)                      # default model at its tuned threshold
pd.DataFrame(pred["rows"]).assign(actual=pd.read_csv(example)[M.TARGET].map({0: "legit", 1: "fraud"}))

In [ ]:
from handler import EndpointHandler                    # model/handler.py (Inference Endpoints entry point)

endpoint = EndpointHandler(str(model_dir))
csv_text = example.read_text(encoding="utf-8")
out = endpoint({"inputs": csv_text, "parameters": {"threshold": 0.5}})
print({k: v for k, v in out.items() if k != "rows"})
rows = pd.read_csv(example).head(3).to_dict(orient="records")
print(endpoint({"inputs": {"rows": rows}, "parameters": {"model": "keras_mlp"}})["rows"])

## 8. Export
`export.export` writes `metrics.json` (test metrics of the default model + every variant), the experiment
graphs in `assets/` and refreshes the model card (`README.md`: YAML `model-index`, metrics table, experiment
table). `cfg.model_dir` is then a complete Hugging Face model repo. Set `PUSH_TO_HUB = True` in section 2
after a full run to upload it; the token comes from `hf auth login` or an `HF_TOKEN` secret / environment
variable. Smoke runs are never uploaded.

In [ ]:
metrics = export.export(cfg, best=best, exported=exported, evals=evals, splits=splits,
                        history=history.history, device=device.lower(), train_time_s=train_time_s)
print("test", metrics["primary_metric"])

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():<30} {path.stat().st_size / 1e3:>9.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))